In [3]:
import kagglehub
import boto3
import os

# download dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")

print(path)

C:\Users\DELL\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5


In [11]:
import kagglehub
import boto3
import os

# Download dataset
path = kagglehub.dataset_download("shivamb/netflix-shows")

print("Dataset downloaded to:", path)

# Locate CSV file
csv_file = os.path.join(path, "netflix_titles.csv")

print("CSV File:", csv_file)

# Create S3 client
s3 = boto3.client("s3")

# Your bucket name
bucket_name = "netflix-validation"

# Upload file to S3
s3.upload_file(
     csv_file,
     bucket_name,
     "Raw-files/netflix_titles.csv"
 )

# print("File uploaded successfully!")

Dataset downloaded to: C:\Users\DELL\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5
CSV File: C:\Users\DELL\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5\netflix_titles.csv


In [13]:
import pandas as pd
df = pd.read_csv(path + "/netflix_titles.csv")
df.head()
df.drop_duplicates(inplace=True)

In [20]:
print("Shape of the data set")
print(df.shape)

print("\nColumns present in the data set")
print(df.columns)

print("\nInformation of the data set")
df.info()

Shape of the data set
(8807, 12)

Columns present in the data set
Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

Information of the data set
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.9 MB


In [23]:
df.to_parquet("netflix.parquet")

In [24]:
import snowflake.connector

conn = snowflake.connector.connect(
    user='tanishkarajputt',
    password='tRxRKwxuswZ9mex',
    account='ve57892.ap-southeast-7.aws'
)

print("Connected!")

Connected!


In [26]:
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS netflix_db")
cursor.execute("USE DATABASE netflix_db")

cursor.execute("CREATE SCHEMA IF NOT EXISTS raw")

print("Database and schema created!")

Database and schema created!


In [28]:
cursor.execute("""
CREATE WAREHOUSE IF NOT EXISTS netflix_wh
WITH WAREHOUSE_SIZE='XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE
""")

cursor.execute("USE WAREHOUSE netflix_wh")

print("Warehouse ready!")

Warehouse ready!


In [30]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("obaidhere/netflix-content-strategy-genre-and-rating-analysis")

print("Dataset folder path:", path)
print("Files inside folder:", os.listdir(path))

100%|██████████| 1.34M/1.34M [00:01<00:00, 811kB/s]

Extracting files...
Dataset folder path: C:\Users\DELL\.cache\kagglehub\datasets\obaidhere\netflix-content-strategy-genre-and-rating-analysis\versions\1
Files inside folder: ['netflix_titles.csv']


In [32]:
df = pd.read_csv(path + "/netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [35]:
from snowflake.connector.pandas_tools import write_pandas

# Select database/schema first
cursor.execute("USE DATABASE netflix_db")
cursor.execute("USE SCHEMA raw")

success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df,
    table_name='NETFLIX_RAW',
    auto_create_table=True
)

print(success)
print(nrows)

True
8807
